# AutomixRouter - Training

This notebook demonstrates how to train the **AutomixRouter** (Automatic LLM Mixing Router).

## Overview

AutomixRouter implements cost-efficient routing with self-verification. It starts with a small model
and escalates to a larger model only when needed, based on verification of the response quality.

**Key Features**:
- POMDP-based routing strategy
- Self-consistency verification
- Cost-aware routing
- Automatic small/large model selection

**Routing Methods**:
1. **Threshold**: Simple threshold-based routing
2. **SelfConsistency**: Self-consistency check before escalation
3. **POMDP**: Partially Observable Markov Decision Process

## 1. Environment Setup

In [ ]:
!pip install llmrouter-lib transformers torch peft accelerate bitsandbytes
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
from llmrouter.models.automix import AutomixRouter, AutomixRouterTrainer
from llmrouter.utils import setup_environment

setup_environment()
print("Environment setup complete!")

## 2. Configuration

AutomixRouter uses the following configuration parameters:

| Parameter | Description | Default |
|-----------|-------------|--------|
| `routing_method` | Routing strategy | "POMDP" |
| `num_bins` | Discretization bins | 8 |
| `small_model_cost` | Cost of small model | 1 |
| `large_model_cost` | Cost of large model | 50 |
| `verifier_cost` | Cost of verifier | 1 |

In [ ]:
import yaml

CONFIG_PATH = "configs/model_config_train/automix.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

## 3. Initialize Router

In [ ]:
router = AutomixRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Number of training samples: {len(router.routing_data_train)}")
print(f"Routing method: {config['hparam']['routing_method']}")

## 4. Training

In [ ]:
trainer = AutomixRouterTrainer(router=router, device='cpu')

print("Trainer initialized!")

In [ ]:
print("Starting training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training completed!")

## 5. Model Verification

In [ ]:
# Test prediction
test_query = {"query": "What is 2 + 2?"}
result = router.route_single(test_query)

print(f"Test query: {test_query['query']}")
print(f"Routed to: {result['model_name']}")

## Summary

In this notebook, we:

1. **Loaded Configuration**: Set up AutomixRouter with YAML configuration
2. **Trained Model**: Learned POMDP-based routing policy
3. **Verified Model**: Tested routing with sample queries

**Key Takeaways**:
- AutomixRouter optimizes cost-performance tradeoff
- Self-verification helps avoid unnecessary escalations
- POMDP provides optimal policy under uncertainty

**Next Steps**:
- Use next part of notebook for inference

# AutomixRouter - Inference

This part of notebook demonstrates how to use a trained **AutomixRouter** for inference.

## 1. Environment Setup (optional)

In [ ]:
!pip install llmrouter-lib transformers torch peft accelerate bitsandbytes
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

## 2. Load Trained Router

In [ ]:
CONFIG_PATH = "configs/model_config_train/automix.yaml"

router = AutomixRouter(yaml_path=CONFIG_PATH)
print("Router loaded!")

## 3. Query Routing

In [ ]:
EXAMPLE_QUERIES = [
    {"query": "What is 2 + 2?"},  # Simple - small model
    {"query": "Solve: 3x^2 - 7x + 2 = 0"},  # Medium complexity
    {"query": "Prove the Riemann Hypothesis."},  # Complex - large model
]

print("Routing Results:")
print("=" * 60)

for i, query in enumerate(EXAMPLE_QUERIES, 1):
    result = router.route_single(query)
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result['model_name']}")

## 4. File-Based Inference

Load queries from a file and save results.

In [ ]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/automix_router_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")
    
    # Save results
    save_results_to_file(file_results, OUTPUT_FILE)
    
    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

## Summary

AutomixRouter intelligently routes:
- Simple queries to small, cheap models
- Complex queries to large, capable models
- Uses self-verification to validate routing decisions